# NetFlow-v3 Feature Pruning, Normalization & Resampling Pipeline
## Production Data Engineering for Cross-Domain IDS & MIA Benchmarking

This notebook implements the end-to-end data processing and resampling pipeline for the **NetFlow-v3 benchmark suite**:
1. **Shortcut Learning Mitigation**: Pruning non-generalizable network identifiers (`IPV4_SRC_ADDR`, `L4_SRC_PORT`, `IPV4_DST_ADDR`, `L4_DST_PORT`, `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS`).
2. **Exact Vectorized Deduplication**: Eliminating synthetic testbed duplicate flows while strictly preserving legitimate traffic distributions.
3. **Continuous Feature Normalization**: Robust Min-Max scaling of all 47 numerical flow features into $[0, 1]$ with finite handling.
4. **Targeted Resampling Strategy (Per Domain)**:
   - **NF-BoT-IoT-v3 (Smart Home)**: **C-VAE + RUS** (Downsampling dominant DoS/DDoS head classes; C-VAE generative oversampling of rare `Theft`).
   - **NF-ToN-IoT-v3 (Hybrid IIoT / Cloud)**: **C-VAE** (Generative oversampling of ultra-rare minority attacks: `ransomware`, `mitm`, `Backdoor`).
   - **NF-UNSW-NB15-v3 (Campus / Enterprise)**: **None** (Preserves natural distribution).
   - **NF-CSE-CIC-IDS2018-v3 (AWS Cloud Enterprise)**: **None** (Preserves natural distribution).
5. **Standardized Export**: Materializing all processed datasets into `NetFlow_IDS_final/`.


## 1. Environment, DuckDB & CVAE Engine Setup


In [ ]:
import os
import sys
import time
from pathlib import Path

# Prevent OpenMP library runtime clash on Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add current path to import CVAEOversampler
curr_dir = Path.cwd()
if str(curr_dir) not in sys.path:
    sys.path.append(str(curr_dir))

from cvae_oversampling import CVAEOversampler

# Initialize DuckDB with high-throughput configuration
con = duckdb.connect()
con.execute("PRAGMA threads=6;")
con.execute("PRAGMA memory_limit='10GB';")
con.execute("PRAGMA preserve_insertion_order=false;")

print(f"DuckDB Version: {duckdb.__version__}")
print(f"PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")


## 2. Dataset Discovery & Shortcut Feature Pruning

Features that act as network identifiers or temporal markers allow tree-based and neural network models to exploit **shortcut learning** (memorizing specific IP addresses, destination ports like 445 or 3389, or time windows) rather than learning invariant flow characteristics.

We explicitly drop the 6 non-generalizable columns:
* `IPV4_SRC_ADDR`, `IPV4_DST_ADDR` (Network Layer IP endpoints)
* `L4_SRC_PORT`, `L4_DST_PORT` (Transport Layer port numbers)
* `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS` (Temporal epoch timestamps)

Remaining schema: **47 standardized continuous/categorical flow features + Label + Attack (49 total columns)**.


In [ ]:
# Locate NetFlow datasets dynamically
base_dir = Path.cwd()
possible_roots = [
    base_dir / '../../datasets/NetFlow',
    base_dir / '../datasets/NetFlow',
    base_dir / 'duc_thanh/datasets/NetFlow',
    Path('d:/Swinburne Courses/Project A/duc_thanh/datasets/NetFlow'),
    Path('d:/Swinburne Courses/Project A/datasets/NetFlow')
]

netflow_root = next((p.resolve() for p in possible_roots if p.exists() and p.is_dir()), None)
if not netflow_root:
    raise FileNotFoundError("Could not find datasets/NetFlow directory!")

print(f"NetFlow Source Directory: {netflow_root}")

# Map dataset names to raw CSV files
dataset_files = {
    'unsw': list(netflow_root.glob('NF-UNSW-NB15-v3/*/data/NF-UNSW-NB15-v3.csv'))[0],
    'bot_iot': list(netflow_root.glob('NF-BoT-IoT-v3/*/data/NF-BoT-IoT-v3.csv'))[0],
    'cic_ids': list(netflow_root.glob('NF-CSE-CIC-IDS2018-v3/*/data/NF-CICIDS2018-v3.csv'))[0],
    'ton_iot': list(netflow_root.glob('NF-ToN-IoT-v3/*/data/NF-ToN-IoT-v3.csv'))[0],
}

# Output directory setup
export_dir = netflow_root.parent / 'NetFlow_IDS_final'
export_dir.mkdir(parents=True, exist_ok=True)
print(f"Export Target Directory:  {export_dir.resolve()}")

EXCLUDE_COLS = [
    'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT',
    'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS'
]
EXCLUDE_SQL = f"EXCLUDE ({', '.join(EXCLUDE_COLS)})"

def get_feature_columns(table_or_view: str) -> list:
    cols = [r[0] for r in con.execute(f"DESCRIBE {table_or_view};").fetchall()]
    return [c for c in cols if c not in ['Label', 'Attack']]

def compute_min_max_expressions(table_name: str, feat_cols: list) -> list:
    agg_exprs = [
        f"MIN(CASE WHEN isfinite({c}::DOUBLE) THEN {c}::DOUBLE ELSE NULL END) AS min_{c}, "
        f"MAX(CASE WHEN isfinite({c}::DOUBLE) THEN {c}::DOUBLE ELSE NULL END) AS max_{c}"
        for c in feat_cols
    ]
    min_max_df = con.execute(f"SELECT {', '.join(agg_exprs)} FROM {table_name};").df()
    scale_exprs = []
    for c in feat_cols:
        c_min = float(min_max_df[f"min_{c}"].iloc[0])
        c_max = float(min_max_df[f"max_{c}"].iloc[0])
        denom = c_max - c_min
        if denom <= 0 or not np.isfinite(denom):
            scale_exprs.append(f"0.0 AS {c}")
        else:
            scale_exprs.append(f'''
                CASE 
                    WHEN NOT isfinite({c}::DOUBLE) THEN 0.0
                    WHEN {c}::DOUBLE <= {c_min} THEN 0.0
                    WHEN {c}::DOUBLE >= {c_max} THEN 1.0
                    ELSE ROUND(({c}::DOUBLE - {c_min}) / {denom}, 6)
                END AS {c}
            '''.strip())
    return scale_exprs


## 3. NF-UNSW-NB15-v3 Pipeline (Resampling: None)


In [ ]:
print(">>> Processing NF-UNSW-NB15-v3 (Resampling: None)...")
t0 = time.time()
raw_path_unsw = str(dataset_files['unsw']).replace('\\', '/')

# 1. Deduplication view
con.execute(f"""
    CREATE OR REPLACE VIEW unsw_clean AS
    SELECT DISTINCT * {EXCLUDE_SQL}
    FROM read_csv_auto('{raw_path_unsw}');
""")

feat_cols_unsw = get_feature_columns('unsw_clean')
scale_exprs_unsw = compute_min_max_expressions('unsw_clean', feat_cols_unsw)

# 2. Materialize normalized table
con.execute(f"""
    CREATE OR REPLACE TABLE unsw_final AS
    SELECT 
        {', '.join(scale_exprs_unsw)},
        Label,
        Attack
    FROM unsw_clean;
""")

# 3. Export to NetFlow_IDS_final
out_unsw = export_dir / 'NF-UNSW-NB15-v3.csv'
con.execute(f"COPY unsw_final TO '{str(out_unsw).replace('\\\\', '/')}' (HEADER, DELIMITER ',');")

n_unsw = con.execute("SELECT COUNT(*) FROM unsw_final").fetchone()[0]
print(f"NF-UNSW-NB15-v3 Export Complete: {n_unsw:,} flows | {out_unsw.stat().st_size / (1024**2):.2f} MB | {time.time() - t0:.2f}s")
con.execute("SELECT Attack, COUNT(*) as cnt FROM unsw_final GROUP BY Attack ORDER BY cnt DESC;").df()


## 4. NF-BoT-IoT-v3 Pipeline (Resampling: C-VAE + RUS)

BoT-IoT presents extreme class imbalance:
* `DoS` and `DDoS` constitute over **95%** of all attack flows.
* Rare classes like `Theft` contain only **448** unique flows.

**Strategy**:
1. **Random Under-Sampling (RUS)**: Downsample majority classes (`DoS`, `DDoS`, `Reconnaissance`) to 25,000 flows each.
2. **Conditional VAE (C-VAE)**: Synthesize high-fidelity flows for `Theft` up to 20,000 samples.
3. **Normalization**: Min-Max scale all 47 features to $[0, 1]$.


In [ ]:
print(">>> Processing NF-BoT-IoT-v3 (Resampling: C-VAE + RUS)...")
t0 = time.time()
raw_path_bot = str(dataset_files['bot_iot']).replace('\\', '/')

# 1. Deduplication
con.execute(f"""
    CREATE OR REPLACE VIEW bot_clean AS
    SELECT DISTINCT * {EXCLUDE_SQL}
    FROM read_csv_auto('{raw_path_bot}');
""")

feat_cols_bot = get_feature_columns('bot_clean')

# 2. Random Under-Sampling (RUS)
con.execute("""
    CREATE OR REPLACE TABLE bot_rus AS
    (SELECT * FROM bot_clean WHERE Attack = 'DoS' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'DDoS' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'Reconnaissance' USING SAMPLE 25000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'Benign')
    UNION ALL
    (SELECT * FROM bot_clean WHERE Attack = 'Theft');
""")

# 3. C-VAE Generative Oversampling for 'Theft'
df_bot_attacks = con.execute("SELECT * FROM bot_rus WHERE Label = 1;").df()
cvae_bot = CVAEOversampler(latent_dim=16, epochs=15, batch_size=128, beta=0.5, random_state=42)
cvae_bot.fit(df_bot_attacks, target_col='Attack', feature_cols=feat_cols_bot, verbose=False)

current_theft = (df_bot_attacks['Attack'] == 'Theft').sum()
needed_theft = 20000 - current_theft
df_syn_theft = cvae_bot.generate_samples('Theft', needed_theft)
df_syn_theft['Label'] = 1
df_syn_theft['Attack'] = 'Theft'
df_syn_theft = df_syn_theft[feat_cols_bot + ['Label', 'Attack']]

con.register('bot_syn_view', df_syn_theft)
con.execute("""
    CREATE OR REPLACE TABLE bot_resampled AS
    SELECT * FROM bot_rus
    UNION ALL BY NAME
    SELECT * FROM bot_syn_view;
""")

# 4. Normalization & Export
scale_exprs_bot = compute_min_max_expressions('bot_resampled', feat_cols_bot)
con.execute(f"""
    CREATE OR REPLACE TABLE bot_final AS
    SELECT 
        {', '.join(scale_exprs_bot)},
        Label,
        Attack
    FROM bot_resampled;
""")

out_bot = export_dir / 'NF-BoT-IoT-v3.csv'
con.execute(f"COPY bot_final TO '{str(out_bot).replace('\\\\', '/')}' (HEADER, DELIMITER ',');")

n_bot = con.execute("SELECT COUNT(*) FROM bot_final").fetchone()[0]
print(f"NF-BoT-IoT-v3 Export Complete: {n_bot:,} flows | {out_bot.stat().st_size / (1024**2):.2f} MB | {time.time() - t0:.2f}s")
con.execute("SELECT Attack, COUNT(*) as cnt FROM bot_final GROUP BY Attack ORDER BY cnt DESC;").df()


## 5. NF-ToN-IoT-v3 Pipeline (Resampling: C-VAE)

ToN-IoT contains minority attack classes (`ransomware`, `mitm`, `Backdoor`) that are heavily underrepresented.

**Strategy**:
1. Train C-VAE on attack representations.
2. Synthesize minority classes (`ransomware`, `mitm`, `Backdoor`) up to 25,000 flows each.
3. Min-Max normalize all features into $[0, 1]$ and export.


In [ ]:
print(">>> Processing NF-ToN-IoT-v3 (Resampling: C-VAE)...")
t0 = time.time()
raw_path_ton = str(dataset_files['ton_iot']).replace('\\', '/')

# 1. Deduplication
con.execute(f"""
    CREATE OR REPLACE VIEW ton_clean AS
    SELECT DISTINCT * {EXCLUDE_SQL}
    FROM read_csv_auto('{raw_path_ton}');
""")

feat_cols_ton = get_feature_columns('ton_clean')

# 2. C-VAE Training
q_ton_train = """
    (SELECT * FROM ton_clean WHERE Label = 1 USING SAMPLE 30000 (reservoir, 42))
    UNION ALL
    (SELECT * FROM ton_clean WHERE Attack IN ('mitm', 'ransomware', 'Backdoor'))
"""
df_ton_train = con.execute(q_ton_train).df().drop_duplicates()

cvae_ton = CVAEOversampler(latent_dim=24, epochs=15, batch_size=128, beta=0.5, random_state=42)
cvae_ton.fit(df_ton_train, target_col='Attack', feature_cols=feat_cols_ton, verbose=False)

# 3. Synthesize minority classes
current_counts = con.execute("""
    SELECT Attack, COUNT(*) as cnt 
    FROM ton_clean 
    WHERE Attack IN ('ransomware', 'mitm', 'Backdoor')
    GROUP BY Attack;
""").df().set_index('Attack')['cnt'].to_dict()

rare_targets = {'ransomware': 25000, 'mitm': 25000, 'Backdoor': 25000}
syn_dfs = []
for cls_name, target_cnt in rare_targets.items():
    curr = current_counts.get(cls_name, 0)
    needed = target_cnt - curr
    if needed > 0:
        syn = cvae_ton.generate_samples(cls_name, needed)
        syn['Label'] = 1
        syn['Attack'] = cls_name
        syn = syn[feat_cols_ton + ['Label', 'Attack']]
        syn_dfs.append(syn)

df_ton_syn = pd.concat(syn_dfs, ignore_index=True)
con.register('ton_syn_view', df_ton_syn)

con.execute("""
    CREATE OR REPLACE TABLE ton_resampled AS
    SELECT * FROM ton_clean
    UNION ALL BY NAME
    SELECT * FROM ton_syn_view;
""")

# 4. Normalization & Export
scale_exprs_ton = compute_min_max_expressions('ton_resampled', feat_cols_ton)
con.execute(f"""
    CREATE OR REPLACE TABLE ton_final AS
    SELECT 
        {', '.join(scale_exprs_ton)},
        Label,
        Attack
    FROM ton_resampled;
""")

out_ton = export_dir / 'NF-ToN-IoT-v3.csv'
con.execute(f"COPY ton_final TO '{str(out_ton).replace('\\\\', '/')}' (HEADER, DELIMITER ',');")

n_ton = con.execute("SELECT COUNT(*) FROM ton_final").fetchone()[0]
print(f"NF-ToN-IoT-v3 Export Complete: {n_ton:,} flows | {out_ton.stat().st_size / (1024**2):.2f} MB | {time.time() - t0:.2f}s")
con.execute("SELECT Attack, COUNT(*) as cnt FROM ton_final GROUP BY Attack ORDER BY cnt DESC;").df()


## 6. NF-CSE-CIC-IDS2018-v3 Pipeline (Resampling: None)


In [ ]:
print(">>> Processing NF-CSE-CIC-IDS2018-v3 (Resampling: None)...")
t0 = time.time()
raw_path_cic = str(dataset_files['cic_ids']).replace('\\', '/')

# 1. Deduplication view
con.execute(f"""
    CREATE OR REPLACE VIEW cic_clean AS
    SELECT DISTINCT * {EXCLUDE_SQL}
    FROM read_csv_auto('{raw_path_cic}');
""")

feat_cols_cic = get_feature_columns('cic_clean')
scale_exprs_cic = compute_min_max_expressions('cic_clean', feat_cols_cic)

# 2. Materialize normalized table
con.execute(f"""
    CREATE OR REPLACE TABLE cic_final AS
    SELECT 
        {', '.join(scale_exprs_cic)},
        Label,
        Attack
    FROM cic_clean;
""")

# 3. Export to NetFlow_IDS_final
out_cic = export_dir / 'NF-CSE-CIC-IDS2018-v3.csv'
con.execute(f"COPY cic_final TO '{str(out_cic).replace('\\\\', '/')}' (HEADER, DELIMITER ',');")

n_cic = con.execute("SELECT COUNT(*) FROM cic_final").fetchone()[0]
print(f"NF-CSE-CIC-IDS2018-v3 Export Complete: {n_cic:,} flows | {out_cic.stat().st_size / (1024**2):.2f} MB | {time.time() - t0:.2f}s")
con.execute("SELECT Attack, COUNT(*) as cnt FROM cic_final GROUP BY Attack ORDER BY cnt DESC;").df()


## 7. Export Catalog Verification & Summary Diagnostics


In [ ]:
summary_records = []
for p in sorted(export_dir.glob('*.csv')):
    t_cnt = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{str(p).replace('\\\\', '/')}')").fetchone()[0]
    size_mb = p.stat().st_size / (1024**2)
    summary_records.append({
        'Dataset': p.stem,
        'Filename': p.name,
        'Flow Count': f"{t_cnt:,}",
        'File Size (MB)': round(size_mb, 2)
    })

df_summary = pd.DataFrame(summary_records)
print("=== Final Exported Datasets in NetFlow_IDS_final ===")
display(df_summary)
